<a href="https://colab.research.google.com/github/qhfl3644/team10-recsys/blob/main/notebooks/02b_dataset_overlap.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02b. 두 데이터셋의 사용자·상품 중복 (보고서 2.4절) — 최지훈 A 업무

구축용 `Video_Games`와 테스트용 `Toys_and_Games`에 같은 사용자·상품이 얼마나 함께 나오는지 센다.
원본 ID(`user_id`, `item_id` = 원본 `parent_asin`)를 그대로 비교한다. 팀 번호(`uidx`, `iidx`)는 데이터셋마다 따로 매긴 번호라 쓰지 않는다.

| 셀 | 내용 | 출력 (Drive `results/eda/`) |
|---|---|---|
| 0~1 | 설정, 데이터 읽기 | |
| 2 | 사용자·상품 중복 요약 | `overlap_summary.csv` |
| 3 | 공통 상품 목록 (있을 때만) | `overlap_common_items.csv` |
| 4 | 공통 사용자의 구간 교차표 + 02_eda와 구간 크기 점검 | `overlap_user_groups.csv` |
| 5 | 평가 표본 안의 공통 사용자 비율 (전체·구간별, 프로토콜 ③ 표본) | `overlap_share.csv` |
| 6 | 공통 사용자 vs 나머지, 사용자별 평균 평점 상관 (참고) | `overlap_users_profile.csv`, `overlap_user_mean_corr.json` |
| 7 | 보고서 2.4절용 요약 출력 | |

- 필요한 것: `01_data_prep` 결과(`processed/`). `03_sampling`의 `p3_users`가 있으면 ③ 표본 비율도 계산한다
- 코드: `src/data/overlap.py`. 예상 시간 1~3분 (Toys 평점 386만 행 읽기 포함)
- 실행 결과는 Colab **파일 → GitHub에 사본 저장**에서 경로를 `notebooks/runs/02b_dataset_overlap_run.ipynb`로 바꿔 저장한다

In [ ]:
# ===== 0. 설정 =====
import os, sys, subprocess, json, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO = '/content/team10-recsys'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', 'https://github.com/qhfl3644/team10-recsys.git', REPO], check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)
except ImportError:
    REPO = os.environ.get('TEAM10_REPO', '..')
sys.path.insert(0, REPO)
from config import *

import numpy as np
import pandas as pd
from src.data import overlap as OV

A, B = DATASETS                      # 구축용, 테스트용
OUT = f'{RESULT_DIR}/eda'            # 02_eda와 같은 폴더
os.makedirs(OUT, exist_ok=True)
print(f'{A} vs {B}, 저장 위치 {OUT}')

## 1. 데이터 읽기

중복은 `ratings_all`(train + valid + test)로 세고, 사용자 구간은 `train` 평점 수로 나눈다 (`metrics.Data.group_of`와 같은 기준).
데이터셋별 수치가 02_eda와 같아야 한다 (Video_Games 평점 814,586 / 사용자 94,762 / 상품 25,612, Toys_and_Games 3,861,886 / 432,264 / 162,035).

In [ ]:
# ===== 1. 데이터 읽기 =====
t0 = time.time()
ratings = {ds: pd.read_parquet(proc_path(ds, 'ratings_all'), columns=['user_id', 'item_id', 'rating']) for ds in DATASETS}
train = {ds: pd.read_parquet(proc_path(ds, 'train'), columns=['user_id']) for ds in DATASETS}
for ds in DATASETS:
    r = ratings[ds]
    print(f'{ds}: 평점 {len(r):,} / 사용자 {r.user_id.nunique():,} / 상품 {r.item_id.nunique():,}')
print(f'({time.time()-t0:.0f}초)')

## 2. 사용자·상품 중복 요약

- `{데이터셋} 대비(%)` = 공통 ID 수 ÷ 그 데이터셋의 고유 ID 수
- `{데이터셋} 평점 중 공통 몫(%)` = 공통 ID가 들어간 평점 행 ÷ 그 데이터셋의 전체 평점 행

In [ ]:
# ===== 2. 중복 요약 =====
summary = OV.overlap_table(ratings)
common_users = OV.common_ids(ratings, 'user_id')
common_items = OV.common_ids(ratings, 'item_id')
assert len(common_users) == int(summary.loc[summary['대상'] == '사용자', '공통'].iloc[0])
assert len(common_items) == int(summary.loc[summary['대상'] == '상품', '공통'].iloc[0])
summary.to_csv(f'{OUT}/overlap_summary.csv', index=False, encoding='utf-8-sig')
print(summary.round(2).to_string(index=False))

## 3. 공통 상품

두 카테고리에 모두 나오는 상품이 있으면 제목과 데이터셋별 평점 수를 본다. 없으면 Toys의 상품은 전부 Video_Games에 없는 상품이다.

In [ ]:
# ===== 3. 공통 상품 =====
if len(common_items):
    ci = pd.DataFrame({'item_id': common_items.astype(str)})
    for ds in DATASETS:
        cnt = ratings[ds]['item_id'].astype(str).value_counts()
        ci[f'{ds} 평점 수'] = cnt.reindex(common_items).to_numpy()
    feat = pd.read_parquet(proc_path(A, 'item_features'), columns=['item_id', 'title'])
    feat['item_id'] = feat['item_id'].astype(str)
    ci = ci.merge(feat, on='item_id', how='left').sort_values(f'{A} 평점 수', ascending=False)
    ci.to_csv(f'{OUT}/overlap_common_items.csv', index=False, encoding='utf-8-sig')
    print(f'공통 상품 {len(ci):,}개 (Video_Games 평점 수 많은 순 10개)')
    print(ci.head(10).assign(title=ci['title'].astype(str).str.slice(0, 60)).to_string(index=False))
else:
    print(f'공통 상품 0개 → {B}의 상품은 모두 {A}에 없는 상품')

## 4. 공통 사용자의 구간 교차표

행 = Video_Games 구간, 열 = Toys_and_Games 구간 (train 평점 수 3~4 / 5~9 / 10+). 대각선 밖은 한쪽에서는 cold인데 다른 쪽에서는 아닌 사용자다.
아래 점검은 전체 사용자의 구간 크기가 02_eda 요약표(`eda_summary.csv`)와 같은지 본다.

In [ ]:
# ===== 4. 구간 교차표 =====
ct = OV.group_crosstab(train[A], train[B], common_users, names=(A, B))
ct.to_csv(f'{OUT}/overlap_user_groups.csv', encoding='utf-8-sig')
print(ct.to_string(), '\n')

# 점검: 전체 사용자 구간 크기 = 02_eda 값
EDA_LABEL = {'cold(3-4)': 'train 3~4개 사용자', 'mid(5-9)': 'train 5~9개 사용자', 'active(10+)': 'train 10개+ 사용자'}
eda_path = f'{OUT}/eda_summary.csv'
eda = pd.read_csv(eda_path, encoding='utf-8-sig') if os.path.exists(eda_path) else None
for ds in DATASETS:
    sizes = OV.user_groups(train[ds]).value_counts().reindex(OV.GROUPS, fill_value=0)
    msg = ', '.join(f'{g} {n:,}' for g, n in sizes.items())
    if eda is None:
        msg += '  (eda_summary.csv 없음, 점검 생략)'
    else:
        ref = eda[eda['데이터셋'] == ds].set_index('항목')['우리 값']
        same = all(EDA_LABEL[g] in ref.index and int(round(float(ref[EDA_LABEL[g]]))) == int(n) for g, n in sizes.items())
        msg += '  → 02_eda와 ' + ('같음' if same else '다름! 데이터 버전을 확인하세요')
    print(f'{ds} 전체 사용자: {msg}')

## 5. 평가 표본 안의 공통 사용자 비율

leave-last-out이라 test 평점 1개 = 사용자 1명이다. 그래서 사용자 비율이 곧 test 평점 중 공통 사용자 몫이다.
프로토콜 ③ 고정 사용자 1만 명(`p3_users`)에서도 계산한다.

In [ ]:
# ===== 5. 표본 안의 공통 비율 =====
share = []
for ds in DATASETS:
    s = OV.share_by_group(train[ds], common_users)
    s.insert(0, 'dataset', ds)
    share.append(s)
    p3_path = f'{PROC_DIR}/{ds}/candidates/p3_users.parquet'
    if os.path.exists(p3_path):
        p3 = pd.read_parquet(p3_path, columns=['user_id'])['user_id'].astype(str)
        k = int(p3.isin(common_users).sum())
        share.append(pd.DataFrame([{'dataset': ds, 'group': 'p3_users', '사용자 수': len(p3),
                                    '공통 사용자 수': k, '공통 비율(%)': 100 * k / len(p3)}]))
    else:
        print(f'  (없음) {p3_path} → ③ 표본 비율은 건너뜀')
share = pd.concat(share, ignore_index=True)
share.to_csv(f'{OUT}/overlap_share.csv', index=False, encoding='utf-8-sig')
print(share.round(2).to_string(index=False))

## 6. 공통 사용자의 특징 (참고)

공통 사용자가 나머지보다 활동량이 많은지, 평점을 다르게 주는지 본다.
사용자별 평균 평점의 상관은 후하게 주는 사람·짜게 주는 사람이 카테고리를 넘어 유지되는지 보는 참고 지표다. 판정에는 쓰지 않는다.

In [ ]:
# ===== 6. 공통 사용자 vs 나머지 =====
prof = OV.common_vs_rest(ratings, common_users)
prof.to_csv(f'{OUT}/overlap_users_profile.csv', index=False, encoding='utf-8-sig')
print(prof.round(3).to_string(index=False))

corr = OV.user_mean_corr(ratings[A], ratings[B], common_users)
corr.update(dataset_a=A, dataset_b=B)
with open(f'{OUT}/overlap_user_mean_corr.json', 'w', encoding='utf-8') as f:
    json.dump(corr, f, ensure_ascii=False, indent=1)
print(f"\n공통 사용자 {corr['n_users']:,}명의 사용자별 평균 평점: {A} {corr['mean_rating_a']:.3f} / {B} {corr['mean_rating_b']:.3f}")
print(f"두 데이터셋 사이 상관: Pearson {corr['pearson']:.3f}, Spearman {corr['spearman']:.3f}")

## 7. 보고서 2.4절용 요약

In [ ]:
# ===== 7. 보고서용 요약 =====
s = summary.set_index('대상')
for who in ['사용자', '상품']:
    print(f"공통 {who} {int(s.loc[who, '공통']):,} = {A}의 {s.loc[who, f'{A} 대비(%)']:.2f}%, {B}의 {s.loc[who, f'{B} 대비(%)']:.2f}%"
          f"  (평점 기준 {A} {s.loc[who, f'{A} 평점 중 공통 몫(%)']:.2f}%, {B} {s.loc[who, f'{B} 평점 중 공통 몫(%)']:.2f}%)")
for _, r in share[share['group'] == 'p3_users'].iterrows():
    print(f"프로토콜 ③ 표본 {r['dataset']}: {int(r['사용자 수']):,}명 중 공통 사용자 {int(r['공통 사용자 수']):,}명 ({r['공통 비율(%)']:.2f}%)")
print('\n저장 파일:', sorted(f for f in os.listdir(OUT) if f.startswith('overlap_')))

## 보고서에 쓸 때

- 팀 번호(`uidx`, `iidx`)를 데이터셋마다 따로 매기고 모델도 데이터셋마다 처음부터 학습했으므로, 공통 사용자·상품이 있어도 Video_Games에서 학습한 값(편향, 잠재요인, TF-IDF 어휘)은 Toys로 넘어가지 않는다. 기준 ④는 "같은 설정으로 다른 데이터에서 다시 학습"한 결과다.
- 공통 사용자 비율이 크면 Toys 결과를 "다른 사람들"에 대한 일반화로만 읽을 수 없다. 실제 비율을 2.4절에 적고, 필요하면 한계(5.19절)에 남긴다.
- 6번 셀의 상관은 참고 지표다. 결과 해석에 쓸 때는 가설로 적는다.